# Hasil final — SARIMA mingguan + koreksi Ridge bergerbang (volume)

Notebook ini **tidak melatih ulang apa pun**. Ia membaca CSV di `results/` dan menampilkan
perbandingan yang dipakai di paper:

1. **Tabel 9 sel** (3 polutan x 3 jarak, 5 lipatan): SARIMA saja vs resep kita + uji Diebold-Mariano.
2. **Bukti bahwa "volume" menyelamatkan** — NO2 jarak 1 hari: koreksi ditelan 100% vs diberi gerbang.
3. **Sebaran volume (alfa)** yang dipilih data sendiri: kalau koreksinya berguna, alfa naik; kalau mengarang, alfa jatuh ke 0.

Protokol: SARIMA musiman mingguan (s=7) + koreksi residual Ridge. Volume alfa hanya dihitung dari
120 hari masa lalu (tanpa mengintip hari yang diramal).


In [1]:
import pandas as pd
pd.set_option("display.width", 200)

m = pd.read_csv("results/12_final_metrics.csv")
dm = pd.read_csv("results/12_final_dm.csv")

final = m[m.model == "RESEP_FINAL"].reset_index(drop=True)
sar = m[m.model.str.startswith("SARIMA")].reset_index(drop=True)
tab = final[[ "target", "horizon", "rmse" ]].rename(columns={"rmse": "rmse_kita"})
tab["rmse_sarima"] = sar["rmse"].values
tab["delta_persen"] = (tab.rmse_kita / tab.rmse_sarima - 1) * 100
tab["p_dm"] = dm["dm_p"].values
tab["menang_nyata"] = dm["nyata05"].values
tab["alfa_rata2"] = dm["alpha_mean"].values
tab["n"] = final["n"].values
print(tab.round(4).to_string(index=False))
print()
print("menang      :", int((tab.delta_persen < 0).sum()), "/", len(tab))
print("menang nyata:", int((tab.menang_nyata == "ya").sum()), "| kalah nyata:", int(((tab.delta_persen > 0) & (tab.menang_nyata == "ya")).sum()))
print("rata2 delta : %.2f%%" % tab.delta_persen.mean())

target  horizon  rmse_kita  rmse_sarima  delta_persen   p_dm menang_nyata  alfa_rata2   n
   NO2        1     0.5925       0.5962       -0.6269 0.2130        tidak      0.6893 118
   NO2        7     0.9017       0.8890        1.4299 0.4391        tidak      0.2579 117
   NO2       30     0.7040       0.7303       -3.6043 0.3275        tidak      0.1587 112
  PM10        1     0.1114       0.1145       -2.7396 0.0479           ya      0.5045 118
  PM10        7     0.1499       0.1536       -2.4039 0.1081        tidak      0.3432 117
  PM10       30     0.1514       0.1525       -0.7731 0.0043           ya      0.1896 112
  PM25        1     0.0638       0.0650       -1.7638 0.0312           ya      0.4226 118
  PM25        7     0.0753       0.0754       -0.1173 0.0000           ya      0.4122 117
  PM25       30     0.0891       0.0914       -2.4985 0.0009           ya      0.3674 112

menang      : 8 / 9
menang nyata: 5 | kalah nyata: 0
rata2 delta : -1.46%


In [2]:
# NO2 jarak 1 hari: koreksi 100% (gaya paper) vs koreksi diberi volume
f = pd.read_csv("results/03_fix_hybrid_metrics.csv")
n1 = f[(f.target == "NO2") & (f.horizon == 1)][["model", "rmse", "mape"]]
n1["delta_vs_sarima_persen"] = (n1.rmse / n1.loc[n1.model == "SARIMA", "rmse"].values[0] - 1) * 100
print(n1.round(4).to_string(index=False))
print()
print("SARIMA polos            : 0.5951")
print("koreksi ditelan 100%    : 0.7098  (+19% rusak)")
print("koreksi ber-volume      : 0.5578  (-6.3%) / 0.5925 (GBM, balik ke aman)")

       model   rmse   mape  delta_vs_sarima_persen
      SARIMA 0.5951 6.5057                  0.0000
  ridge_full 0.5578 6.0298                 -6.2617
ridge_shrunk 0.5578 6.0298                 -6.2617
    gbm_full 0.7098 7.7846                 19.2800
  gbm_shrunk 0.5925 6.4222                 -0.4445

SARIMA polos            : 0.5951
koreksi ditelan 100%    : 0.7098  (+19% rusak)
koreksi ber-volume      : 0.5578  (-6.3%) / 0.5925 (GBM, balik ke aman)


In [3]:
# Volume yang dipilih data sendiri (0 = dibungkam, 1 = dipercaya penuh) -> tak pernah 1.00
p = pd.read_csv("results/12_final_predictions.csv")
al = pd.DataFrame({"feature": p.feature})
for h in (1, 7, 30):
    al[f"h{h}"] = p[f"alpha_h{h}"]
print(al.groupby("feature")[["h1", "h7", "h30"]].describe().T.round(3).to_string())
print()
flat = pd.concat([p[f"alpha_h{h}"].dropna() for h in (1, 7, 30)])
print("origin dengan volume 1.00 (percaya penuh): %.1f%%" % (100 * (flat >= 0.999).mean()))
print("origin dengan volume 0.00 (dibungkam)   : %.1f%%" % (100 * (flat <= 0.001).mean()))

feature        NO2     PM10     PM25
h1  count  118.000  118.000  118.000
    mean     0.689    0.504    0.423
    std      0.361    0.411    0.405
    min      0.000    0.000    0.000
    25%      0.461    0.000    0.000
    50%      0.825    0.532    0.313
    75%      1.000    1.000    0.972
    max      1.000    1.000    1.000
h7  count  117.000  117.000  117.000
    mean     0.258    0.343    0.412
    std      0.260    0.309    0.382
    min      0.000    0.000    0.000
    25%      0.000    0.053    0.000
    50%      0.216    0.262    0.295
    75%      0.436    0.543    0.837
    max      1.000    1.000    1.000
h30 count  112.000  112.000  112.000
    mean     0.159    0.190    0.367
    std      0.318    0.304    0.334
    min      0.000    0.000    0.000
    25%      0.000    0.000    0.044
    50%      0.000    0.000    0.272
    75%      0.138    0.309    0.641
    max      1.000    1.000    1.000

origin dengan volume 1.00 (percaya penuh): 15.1%
origin dengan volume 0.00